### work on the first love notes

In [ ]:
# query all the notes in the first love deck, which have an empty lemma field. 
# put everything inside 'Word Definition' into 'Notes'# 
# extract 'Lemma' and 'Word Definition' from the cloze (cloze looks like this “人生は{{c1::まるで::as if, quite}} ジグソーパズルだ”と)


In [ ]:
#!/usr/bin/env python3
import json
import re
import urllib.request

ANKI_CONNECT_URL = "http://localhost:8765"
DECK_NAME = "Japanese Media::Filme & Serien::FirstLove"  # change if needed
DRY_RUN = True  # set False to actually write

# lemma: after {{c1:: and before ::
# word definition: after :: and before }}
CLOZE_RE = re.compile(r"\{\{c1::(.*?)::(.*?)\}\}", re.DOTALL)


def invoke(action, **params):
    payload = {"action": action, "version": 6, "params": params}
    req = urllib.request.Request(
        ANKI_CONNECT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        out = json.loads(resp.read().decode("utf-8"))
    if out.get("error"):
        raise RuntimeError(f"{action} failed: {out['error']}")
    return out["result"]


def extract_from_cloze(cloze_text: str):
    m = CLOZE_RE.search(cloze_text or "")
    if not m:
        return None, None
    lemma = m.group(1).strip()
    word_def = m.group(2).strip()
    return lemma, word_def


def merge_notes(notes_val: str, word_def_val: str):
    notes_val = (notes_val or "").strip()
    word_def_val = (word_def_val or "").strip()
    if not word_def_val:
        return notes_val
    if not notes_val:
        return word_def_val
    if word_def_val in notes_val:
        return notes_val
    return f"{notes_val}<br>{word_def_val}"


def main():
    # empty Lemma field
    note_ids = invoke("findNotes", query=f'deck:"{DECK_NAME}" "Lemma:"')
    print(f"Found {len(note_ids)} notes")

    if not note_ids:
        return

    notes = invoke("notesInfo", notes=note_ids)
    updates = 0

    for note in notes:
        note_id = note["noteId"]
        f = note["fields"]

        cloze = f.get("Cloze", {}).get("value", "")
        notes_old = f.get("Notes", {}).get("value", "")
        wd_old = f.get("Word Definition", {}).get("value", "")

        lemma_new, wd_new = extract_from_cloze(cloze)
        notes_new = merge_notes(notes_old, wd_old)

        fields_to_update = {}
        if lemma_new:
            fields_to_update["Lemma"] = lemma_new
        if wd_new is not None:
            fields_to_update["Word Definition"] = wd_new
        if notes_new != notes_old:
            fields_to_update["Notes"] = notes_new

        if not fields_to_update:
            continue

        print(f"Note {note_id} -> {fields_to_update}")
        if not DRY_RUN:
            invoke("updateNoteFields", note={"id": note_id, "fields": fields_to_update})
        updates += 1

    print(f"Done. {updates} notes {'would be updated' if DRY_RUN else 'updated'}.")


if __name__ == "__main__":
    main()


Gefunden: 37 Notizen
Skip 1734510465583: keine passende c1-Cloze gefunden
Skip 1734593383376: keine passende c1-Cloze gefunden
Skip 1734681071354: keine passende c1-Cloze gefunden
Skip 1734681071371: keine passende c1-Cloze gefunden
Skip 1734681071390: keine passende c1-Cloze gefunden
Skip 1734681071416: keine passende c1-Cloze gefunden
Skip 1734681071450: keine passende c1-Cloze gefunden
Skip 1739865122051: keine passende c1-Cloze gefunden
Skip 1739865122055: keine passende c1-Cloze gefunden
Skip 1739865122061: keine passende c1-Cloze gefunden
Skip 1739865122064: keine passende c1-Cloze gefunden
Skip 1739865122065: keine passende c1-Cloze gefunden
Skip 1739865122066: keine passende c1-Cloze gefunden
Skip 1739865122071: keine passende c1-Cloze gefunden
Skip 1739865122075: keine passende c1-Cloze gefunden
Skip 1739865122080: keine passende c1-Cloze gefunden
Skip 1739865122081: keine passende c1-Cloze gefunden
Skip 1739865122082: keine passende c1-Cloze gefunden
Skip 1739865122093: keine

In [ ]:
#!/usr/bin/env python3
import json
import re
import urllib.request

ANKI_CONNECT_URL = "http://localhost:8765"
DECK_NAME = "Japanese Media::Filme & Serien::FirstLove"  # ggf. anpassen
DRY_RUN = True  # erst prüfen, dann auf False setzen

# Beispiel: 人生は{{c1::まるで::as if, quite}} ジグソーパズルだ
# Lemma = zwischen {{c1:: und ::
# Word Definition = zwischen :: und }}
CLOZE_RE = re.compile(r"\{\{c1::(.*?)::(.*?)\}\}", re.DOTALL)


def invoke(action, **params):
    payload = {"action": action, "version": 6, "params": params}
    req = urllib.request.Request(
        ANKI_CONNECT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        out = json.loads(resp.read().decode("utf-8"))
    if out.get("error"):
        raise RuntimeError(f"{action} failed: {out['error']}")
    return out["result"]


def extract_from_cloze(cloze_text: str):
    m = CLOZE_RE.search(cloze_text or "")
    if not m:
        return None, None
    lemma = m.group(1).strip()
    definition = m.group(2).strip()
    return lemma, definition


def backup_word_definition_to_notes(notes_val: str, word_def_old: str):
    notes_val = (notes_val or "").strip()
    word_def_old = (word_def_old or "").strip()

    if not word_def_old:
        return notes_val
    if not notes_val:
        return word_def_old
    if word_def_old in notes_val:
        return notes_val
    return f"{notes_val}<br>{word_def_old}"


def main():
    # Nur Notes im FirstLove-Deck mit leerem Lemma
    note_ids = invoke("findNotes", query=f'deck:"{DECK_NAME}" "Lemma:"')
    print(f"Gefunden: {len(note_ids)} Notizen")

    if not note_ids:
        return

    notes = invoke("notesInfo", notes=note_ids)
    changed = 0
    skipped = 0

    for note in notes:
        note_id = note["noteId"]
        fields = note.get("fields", {})

        cloze = fields.get("Cloze", {}).get("value", "")
        lemma_old = fields.get("Lemma", {}).get("value", "")
        notes_old = fields.get("Notes", {}).get("value", "")
        word_def_old = fields.get("Word Definition", {}).get("value", "")

        lemma_new, word_def_new = extract_from_cloze(cloze)
        if lemma_new is None and word_def_new is None:
            skipped += 1
            print(f"Skip {note_id}: keine passende c1-Cloze gefunden")
            continue

        # WICHTIG: alten Word-Definition-Inhalt zuerst in Notes sichern
        notes_new = backup_word_definition_to_notes(notes_old, word_def_old)

        fields_to_update = {}

        # Lemma setzen (optional, aber meist gewünscht)
        if lemma_new and lemma_new != lemma_old:
            fields_to_update["Lemma"] = lemma_new

        # Word Definition IMMER mit der aus Cloze überschreiben
        if word_def_new is not None and word_def_new != word_def_old:
            fields_to_update["Word Definition"] = word_def_new

        # Notes nur ändern, wenn Backup was verändert hat
        if notes_new != notes_old:
            fields_to_update["Notes"] = notes_new

        if not fields_to_update:
            skipped += 1
            continue

        print(f"Update {note_id}: {fields_to_update}")
        if not DRY_RUN:
            invoke("updateNoteFields", note={"id": note_id, "fields": fields_to_update})
        changed += 1

    print(f"Fertig. Geändert: {changed}, Übersprungen: {skipped}, Dry-Run: {DRY_RUN}")


if __name__ == "__main__":
    main()
